In [1]:
from alpha_vantage.timeseries import TimeSeries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import io
import os
import time
import numpy as np

In [2]:
with open("api_key.txt") as file:
    API_key = file.read()

API_key = API_key.strip()
API_key

'N0YVA5RVTIR2D1OL'

In [3]:
pd.options.display.max_columns = None

## Using Alpha vantage python library

In [ ]:
#ts1 = TimeSeries(key=API_key)

In [ ]:
#ts1.get_monthly('AAPL')

({'2026-04-23': {'1. open': '254.0800',
   '2. high': '275.7700',
   '3. low': '245.7000',
   '4. close': '273.4300',
   '5. volume': '665999564'},
  '2026-03-31': {'1. open': '262.4100',
   '2. high': '266.5300',
   '3. low': '245.5100',
   '4. close': '253.7900',
   '5. volume': '900035757'},
  '2026-02-27': {'1. open': '260.0300',
   '2. high': '280.9050',
   '3. low': '255.4500',
   '4. close': '264.1800',
   '5. volume': '988325816'},
  '2026-01-30': {'1. open': '272.2550',
   '2. high': '277.8400',
   '3. low': '243.4200',
   '4. close': '259.4800',
   '5. volume': '1036170325'},
  '2025-12-31': {'1. open': '278.0100',
   '2. high': '288.6200',
   '3. low': '266.9500',
   '4. close': '271.8600',
   '5. volume': '922283649'},
  '2025-11-28': {'1. open': '270.4200',
   '2. high': '280.3800',
   '3. low': '265.3200',
   '4. close': '278.8500',
   '5. volume': '876481453'},
  '2025-10-31': {'1. open': '255.0400',
   '2. high': '277.3200',
   '3. low': '244.0000',
   '4. close': '270.

## Using Alpha vantage api

In [4]:
def return_json(url):
    # replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
    url = url
    r = requests.get(url)
    data = r.json()
    return data

In [5]:
def query_all_statements(ticker,query_func = None):
    """
    """
    if query_func != None:
        function_names= query_func
    else:
        function_names= {   "INCOME_STATEMENT":"annualReports",
                            "BALANCE_SHEET":"annualReports",
                            "CASH_FLOW":"annualReports",
                            "OVERVIEW":"EMPTY",
                            "DIVIDENDS":"data",
                            "SPLITS":"data",
                            "SHARES_OUTSTANDING":"data",
                            "EARNINGS":"annualEarnings",
                            "EARNINGS_ESTIMATES":"estimates"
                            }

    for func in function_names:
        url = f'https://www.alphavantage.co/query?function={func}&symbol={ticker}&apikey={API_key}'
        return_statement = return_json(url)

        # 1. Check for API limit or Error messages
        if "Information" in return_statement:
            print(f"⚠️ API Limit hit on {func}. Skipping...")
            time.sleep(60) # Wait a full minute if limited
            continue

        if func in ["INCOME_STATEMENT","BALANCE_SHEET","CASH_FLOW"]:
            print(f"key is {func}, value is {function_names[func]}")
            df = pd.json_normalize(return_statement.get(function_names[func])).set_index("fiscalDateEnding")
        elif func == "OVERVIEW":
            print(f"key is {func}, value is {function_names[func]}")
            df = pd.json_normalize(return_statement).set_index("Symbol")
        else:
            print(f"key is {func}, value is {function_names[func]}")
            df = pd.json_normalize(return_statement.get(function_names[func]))

        path = f"stocks/{ticker}"

        if not(os.path.exists(path)):
            os.mkdir(path)
        df.to_csv(f"./{path}/{func}.csv")
        time.sleep(15)

In [6]:
dict_to_pass = {"EARNINGS":"annualEarnings",
                "EARNINGS_ESTIMATES":"estimates"
            }
query_all_statements("NVDA",dict_to_pass)

key is EARNINGS, value is annualEarnings
key is EARNINGS_ESTIMATES, value is estimates


In [40]:
url = 'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=AAPL&apikey={API_key}'
appl_sentiment = return_json(url)

In [41]:
appl_sentiment

{'items': '50',
 'sentiment_score_definition': 'x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish',
 'relevance_score_definition': '0 < x <= 1, with a higher score indicating higher relevance.',
 'feed': [{'title': 'Why Is TD Cowen Doubling Down on Amazon (AMZN), Microsoft (MSFT), and Apple (AAPL) ahead of Earnings Next Week',
   'url': 'https://www.tipranks.com/news/why-is-td-cowen-doubling-down-on-amazon-amzn-microsoft-msft-and-apple-aapl-ahead-of-earnings-next-week',
   'time_published': '20260424T153952',
   'authors': ['Radhika Saraogi'],
   'summary': "TD Cowen is bullish on Amazon, Apple, and Microsoft ahead of their upcoming earnings reports, citing accelerated AI adoption, robust cloud demand, and strong product cycles as key drivers for durable growth. The firm believes the market is underestimating the AI momentum of these Big Tech giants and foresees significant revenue streams from G

### Merge Overview statement

In [ ]:
df1 = pd.read_csv('./stocks/APPL/OVERVIEW.csv')
df1.head()

,Unnamed: 0,date,shares_outstanding_diluted,shares_outstanding_basic,stock_id
0,0,2025-12-31,14810356000,14810356000,1
1,1,2025-09-30,15004697000,15004697000,1
2,2,2025-06-30,14948179000,14948179000,1
3,3,2025-06-28,14948179000,14902886000,1
4,4,2025-03-29,15056133000,14994082000,1


In [ ]:
df2 = pd.read_csv('./stocks/MSFT/OVERVIEW.csv')
df2.head()

,Unnamed: 0,date,shares_outstanding_diluted,shares_outstanding_basic,stock_id
0,0,2025-12-31,7460000000,7460000000,2
1,1,2025-09-30,7466000000,7466000000,2
2,2,2025-06-30,7462000000,7430000000,2
3,3,2025-03-31,7461000000,7434000000,2
4,4,2024-12-31,7468000000,7435000000,2


In [ ]:
df3 = pd.read_csv('./stocks/TSLA/OVERVIEW.csv')
df3["stock_id"] = 3
df3.head()

,Unnamed: 0,date,shares_outstanding_diluted,shares_outstanding_basic,stock_id
0,0,2026-03-31,3538000000,3538000000,3
1,1,2025-12-31,3539000000,3539000000,3
2,2,2025-09-30,3526000000,3526000000,3
3,3,2025-06-30,3519000000,3519000000,3
4,4,2025-03-31,3521000000,3218000000,3


In [ ]:
df = pd.concat([df1,df2,df3],ignore_index=True)
df.tail()

,date,shares_outstanding_diluted,shares_outstanding_basic,stock_id
179,2016-12-31,155105000,155105000,3
180,2016-09-30,156935000,148991000,3
181,2015-12-31,131133000,131133000,3
182,2015-09-30,129006000,129006000,3
183,2014-03-31,123472782,123472782,3


In [ ]:
df.rename(columns = {"Currency":"currency_id",
                     "Symbol":"stock_id"},inplace=True)
df.head()


,stock_id,AssetType,Name,Description,CIK,Exchange,currency_id,Country,Sector,Industry,Address,OfficialSite,FiscalYearEnd,LatestQuarter,MarketCapitalization,EBITDA,PERatio,PEGRatio,BookValue,DividendPerShare,DividendYield,EPS,RevenuePerShareTTM,ProfitMargin,OperatingMarginTTM,ReturnOnAssetsTTM,ReturnOnEquityTTM,RevenueTTM,GrossProfitTTM,DilutedEPSTTM,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,AnalystTargetPrice,AnalystRatingStrongBuy,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongSell,TrailingPE,ForwardPE,PriceToSalesRatioTTM,PriceToBookRatio,EVToRevenue,EVToEBITDA,Beta,52WeekHigh,52WeekLow,50DayMovingAverage,200DayMovingAverage,SharesOutstanding,SharesFloat,PercentInsiders,PercentInstitutions,DividendDate,ExDividendDate
0,AAPL,Common Stock,Apple Inc,Apple Inc. is an American multinational techno...,320193,NASDAQ,USD,USA,TECHNOLOGY,CONSUMER ELECTRONICS,"ONE APPLE PARK WAY, CUPERTINO, CA, UNITED STAT...",https://www.apple.com,September,2025-12-31,3979469914000,152901992000,34.35,2.440,6.00,1.03,0.0038,7.89,29.30,0.2700,0.354,0.2440,1.520,435617006000,206157005000,7.89,0.183,0.157,297.71,6,25,15,1,1,34.35,31.75,9.14,45.12,9.19,26.18,1.109,288.35,192.41,260.15,253.64,14681140000,14656035000,1.640,65.225,2026-02-12,2026-02-09
1,MSFT,Common Stock,Microsoft Corporation,Microsoft Corporation is an American multinati...,789019,NASDAQ,USD,USA,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,"ONE MICROSOFT WAY, REDMOND, WA, UNITED STATES,...",https://www.microsoft.com,June,2025-12-31,3157422768000,175258993000,26.60,1.339,52.62,3.48,0.0082,15.97,41.10,0.3900,0.471,0.1490,0.344,305453007000,209498997000,15.97,0.598,0.167,572.67,10,45,3,0,0,26.60,22.12,10.34,8.07,10.22,16.58,1.107,552.24,356.28,394.52,469.71,7425629000,7414788000,0.079,75.883,2026-06-11,2026-05-21
2,TSLA,Common Stock,Tesla Inc,"Tesla, Inc. is an American electric vehicle an...",1318605,NASDAQ,USD,USA,CONSUMER CYCLICAL,AUTO MANUFACTURERS,"1 TESLA ROAD, AUSTIN, TX, UNITED STATES, 78725",https://www.tesla.com,December,2026-03-31,1412227269000,11094000000,344.97,5.190,21.90,NaN,NaN,1.09,30.31,0.0395,0.042,0.0223,0.049,97878999000,18660999000,1.09,0.083,0.158,416.45,5,18,17,6,2,344.97,181.82,14.43,16.91,14.24,115.47,1.915,498.83,271.00,385.48,401.51,3755724000,2815929000,11.121,44.641,NaN,NaN


In [10]:
df["stock_id"] = np.arange(1,df.shape[0]+1)

In [17]:
df = df.replace({'currency_id':{"USD":1},
            'stock_id':{'AAPL':1,
                        'MSFT':2,
                        'TSLA':3}
            })
df.head()

,stock_id,AssetType,Name,Description,CIK,Exchange,currency_id,Country,Sector,Industry,Address,OfficialSite,FiscalYearEnd,LatestQuarter,MarketCapitalization,EBITDA,PERatio,PEGRatio,BookValue,DividendPerShare,DividendYield,EPS,RevenuePerShareTTM,ProfitMargin,OperatingMarginTTM,ReturnOnAssetsTTM,ReturnOnEquityTTM,RevenueTTM,GrossProfitTTM,DilutedEPSTTM,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,AnalystTargetPrice,AnalystRatingStrongBuy,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongSell,TrailingPE,ForwardPE,PriceToSalesRatioTTM,PriceToBookRatio,EVToRevenue,EVToEBITDA,Beta,52WeekHigh,52WeekLow,50DayMovingAverage,200DayMovingAverage,SharesOutstanding,SharesFloat,PercentInsiders,PercentInstitutions,DividendDate,ExDividendDate
0,1,Common Stock,Apple Inc,Apple Inc. is an American multinational techno...,320193,NASDAQ,1,USA,TECHNOLOGY,CONSUMER ELECTRONICS,"ONE APPLE PARK WAY, CUPERTINO, CA, UNITED STAT...",https://www.apple.com,September,2025-12-31,3979469914000,152901992000,34.35,2.440,6.00,1.03,0.0038,7.89,29.30,0.2700,0.354,0.2440,1.520,435617006000,206157005000,7.89,0.183,0.157,297.71,6,25,15,1,1,34.35,31.75,9.14,45.12,9.19,26.18,1.109,288.35,192.41,260.15,253.64,14681140000,14656035000,1.640,65.225,2026-02-12,2026-02-09
1,2,Common Stock,Microsoft Corporation,Microsoft Corporation is an American multinati...,789019,NASDAQ,1,USA,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,"ONE MICROSOFT WAY, REDMOND, WA, UNITED STATES,...",https://www.microsoft.com,June,2025-12-31,3157422768000,175258993000,26.60,1.339,52.62,3.48,0.0082,15.97,41.10,0.3900,0.471,0.1490,0.344,305453007000,209498997000,15.97,0.598,0.167,572.67,10,45,3,0,0,26.60,22.12,10.34,8.07,10.22,16.58,1.107,552.24,356.28,394.52,469.71,7425629000,7414788000,0.079,75.883,2026-06-11,2026-05-21
2,3,Common Stock,Tesla Inc,"Tesla, Inc. is an American electric vehicle an...",1318605,NASDAQ,1,USA,CONSUMER CYCLICAL,AUTO MANUFACTURERS,"1 TESLA ROAD, AUSTIN, TX, UNITED STATES, 78725",https://www.tesla.com,December,2026-03-31,1412227269000,11094000000,344.97,5.190,21.90,NaN,NaN,1.09,30.31,0.0395,0.042,0.0223,0.049,97878999000,18660999000,1.09,0.083,0.158,416.45,5,18,17,6,2,344.97,181.82,14.43,16.91,14.24,115.47,1.915,498.83,271.00,385.48,401.51,3755724000,2815929000,11.121,44.641,NaN,NaN


In [32]:
df.rename(columns={
    "stock_id": "stock_id",
    "AssetType": "asset_type",
    "Name": "name",
    "Description": "description",
    "CIK": "cik",
    "Exchange": "exchange",
    "currency_id": "currency_id",
    "Country": "country",
    "Sector": "sector",
    "Industry": "industry",
    "Address": "address",
    "OfficialSite": "official_site",
    "FiscalYearEnd": "fiscal_year_end",
    "LatestQuarter": "latest_quarter",
    "MarketCapitalization": "market_capitalization",
    "EBITDA": "ebitda",
    "PERatio": "pe_ratio",
    "PEGRatio": "peg_ratio",
    "BookValue": "book_value",
    "DividendPerShare": "dividend_per_share",
    "DividendYield": "dividend_yield",
    "EPS": "eps",
    "RevenuePerShareTTM": "revenue_per_share_ttm",
    "ProfitMargin": "profit_margin",
    "OperatingMarginTTM": "operating_margin_ttm",
    "ReturnOnAssetsTTM": "return_on_assets_ttm",
    "ReturnOnEquityTTM": "return_on_equity_ttm",
    "RevenueTTM": "revenue_ttm",
    "GrossProfitTTM": "gross_profit_ttm",
    "DilutedEPSTTM": "diluted_eps_ttm",
    "QuarterlyEarningsGrowthYOY": "quarterly_earnings_growth_yoy",
    "QuarterlyRevenueGrowthYOY": "quarterly_revenue_growth_yoy",
    "AnalystTargetPrice": "analyst_target_price",
    "AnalystRatingStrongBuy": "analyst_rating_strong_buy",
    "AnalystRatingBuy": "analyst_rating_buy",
    "AnalystRatingHold": "analyst_rating_hold",
    "AnalystRatingSell": "analyst_rating_sell",
    "AnalystRatingStrongSell": "analyst_rating_strong_sell",
    "TrailingPE": "trailing_pe",
    "ForwardPE": "forward_pe",
    "PriceToSalesRatioTTM": "price_to_sales_ratio_ttm",
    "PriceToBookRatio": "price_to_book_ratio",
    "EVToRevenue": "ev_to_revenue",
    "EVToEBITDA": "ev_to_ebitda",
    "Beta": "beta",
    "52WeekHigh": "high_52_week",
    "52WeekLow": "low_52_week",
    "50DayMovingAverage": "moving_average_50_day",
    "200DayMovingAverage": "moving_average_200_day",
    "SharesOutstanding": "shares_outstanding",
    "SharesFloat": "shares_float",
    "PercentInsiders": "percent_insiders",
    "PercentInstitutions": "percent_institutions",
    "DividendDate": "dividend_date",
    "ExDividendDate": "ex_dividend_date"
},inplace=True)

### Merge Shares outstanding

In [ ]:
df1 = pd.read_csv('./stocks/APPL/SHARES_OUTSTANDING.csv')
df1["stock_id"] = 1
df1.head()

In [ ]:
df2 = pd.read_csv('./stocks/MSFT/SHARES_OUTSTANDING.csv')
df2["stock_id"] = 2
df2.head()

In [ ]:
df3 = pd.read_csv('./stocks/TSLA/SHARES_OUTSTANDING.csv')
df3["stock_id"] = 3
df3.head()

In [ ]:
df = pd.concat([df1,df2,df3],ignore_index=True)
df.drop("Unnamed: 0",axis=1,inplace=True)
df.tail()

In [22]:
df.to_csv('../stock_db/data/shares_outstanding.csv',index=False)

### Earnings

In [25]:
df1 = pd.read_csv('./stocks/AAPL/EARNINGS.csv')
df1["stock_id"] = 1
df1.head()

,Unnamed: 0,fiscalDateEnding,reportedEPS,stock_id
0,0,2026-03-31,2.84,1
1,1,2025-09-30,7.47,1
2,2,2024-09-30,6.08,1
3,3,2023-09-30,6.12,1
4,4,2022-09-30,6.11,1


In [26]:
df2 = pd.read_csv('./stocks/MSFT/EARNINGS.csv')
df2["stock_id"] = 2
df2.head()

,Unnamed: 0,fiscalDateEnding,reportedEPS,stock_id
0,0,2026-03-31,7.86,2
1,1,2025-06-30,13.64,2
2,2,2024-06-30,11.81,2
3,3,2023-06-30,9.81,2
4,4,2022-06-30,9.20,2


In [27]:
df3 = pd.read_csv('./stocks/TSLA/EARNINGS.csv')
df3["stock_id"] = 3
df3.head()

,Unnamed: 0,fiscalDateEnding,reportedEPS,stock_id
0,0,2026-03-31,0.41,3
1,1,2025-12-31,1.26,3
2,2,2024-12-31,2.42,3
3,3,2023-12-31,3.13,3
4,4,2022-12-31,4.07,3


In [28]:
df = pd.concat([df1,df2,df3],ignore_index=True)
df.drop("Unnamed: 0",axis=1,inplace=True)
df.tail()

,fiscalDateEnding,reportedEPS,stock_id
74,2014-12-31,0.0300,3
75,2013-12-31,0.0500,3
76,2012-12-31,-0.2100,3
77,2011-12-31,-0.1600,3
78,2010-12-31,-0.0971,3


In [30]:
df.rename(columns={
                    "stock_id": "stock_id",
                    "fiscalDateEnding": "fiscal_date_ending",
                    "reportedEPS": "reported_eps"
                    }, inplace=True
)

In [32]:
df.to_csv('../stock_db/data/earnings.csv',index=False)

### Dividends

In [33]:
df1 = pd.read_csv('./stocks/AAPL/DIVIDENDS.csv')
df1["stock_id"] = 1
df1.head()

,Unnamed: 0,ex_dividend_date,declaration_date,record_date,payment_date,amount,stock_id
0,0,2026-02-09,2026-01-29,2026-02-09,2026-02-12,0.26,1
1,1,2025-11-10,2025-10-30,2025-11-10,2025-11-13,0.26,1
2,2,2025-08-11,2025-07-31,2025-08-11,2025-08-14,0.26,1
3,3,2025-05-12,2025-05-01,2025-05-12,2025-05-15,0.26,1
4,4,2025-02-10,2025-01-30,2025-02-10,2025-02-13,0.25,1


In [34]:
df2 = pd.read_csv('./stocks/MSFT/DIVIDENDS.csv')
df2["stock_id"] = 2
df2.head()

,Unnamed: 0,ex_dividend_date,declaration_date,record_date,payment_date,amount,stock_id
0,0,2026-05-21,2026-03-10,2026-05-21,2026-06-11,0.91,2
1,1,2026-02-19,2025-12-02,2026-02-19,2026-03-12,0.91,2
2,2,2025-11-20,2025-09-15,2025-11-20,2025-12-11,0.91,2
3,3,2025-08-21,2025-06-10,2025-08-21,2025-09-11,0.83,2
4,4,2025-05-15,2025-03-11,2025-05-15,2025-06-12,0.83,2


In [35]:
df3 = pd.read_csv('./stocks/TSLA/DIVIDENDS.csv')
df3["stock_id"] = 3
df3.head()

,Unnamed: 0,stock_id


In [37]:
df = pd.concat([df1,df2],ignore_index=True)
df.drop("Unnamed: 0",axis=1,inplace=True)
df.head()

,ex_dividend_date,declaration_date,record_date,payment_date,amount,stock_id
0,2026-02-09,2026-01-29,2026-02-09,2026-02-12,0.26,1
1,2025-11-10,2025-10-30,2025-11-10,2025-11-13,0.26,1
2,2025-08-11,2025-07-31,2025-08-11,2025-08-14,0.26,1
3,2025-05-12,2025-05-01,2025-05-12,2025-05-15,0.26,1
4,2025-02-10,2025-01-30,2025-02-10,2025-02-13,0.25,1


In [38]:
df.to_csv('../stock_db/data/dividends.csv',index=False)

In [39]:
df.columns

Index(['ex_dividend_date', 'declaration_date', 'record_date', 'payment_date',
       'amount', 'stock_id'],
      dtype='str')

### Splits

In [41]:
df1 = pd.read_csv('./stocks/AAPL/SPLITS.csv')
df1["stock_id"] = 1
df1.head()

,Unnamed: 0,effective_date,split_factor,stock_id
0,0,2020-08-31,4.0,1
1,1,2014-06-09,7.0,1
2,2,2005-02-28,2.0,1
3,3,2000-06-21,2.0,1


In [42]:
df2 = pd.read_csv('./stocks/MSFT/SPLITS.csv')
df2["stock_id"] = 2
df2.head()

,Unnamed: 0,effective_date,split_factor,stock_id
0,0,2003-02-18,2.0,2
1,1,1999-03-29,2.0,2


In [43]:
df3 = pd.read_csv('./stocks/TSLA/SPLITS.csv')
df3["stock_id"] = 3
df3.head()

,Unnamed: 0,effective_date,split_factor,stock_id
0,0,2022-08-25,3.0,3
1,1,2020-08-31,5.0,3


In [44]:
df = pd.concat([df1,df2,df3],ignore_index=True)
df.drop("Unnamed: 0",axis=1,inplace=True)
df.head()

,effective_date,split_factor,stock_id
0,2020-08-31,4.0,1
1,2014-06-09,7.0,1
2,2005-02-28,2.0,1
3,2000-06-21,2.0,1
4,2003-02-18,2.0,2


In [45]:
df.to_csv('../stock_db/data/splits.csv',index=False)

In [46]:
df.columns

Index(['effective_date', 'split_factor', 'stock_id'], dtype='str')

### INCOME Statement

In [52]:
df1 = pd.read_csv('./stocks/AAPL/INCOME_STATEMENT.csv')
df1["stock_id"] = 1
df1.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,stock_id
0,2025-09-30,USD,195201000000,416161000000,220960000000,220960000000,133050000000,8077000000,34550000000,62151000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11698000000,132729000000,20719000000,NaN,1.120100e+11,NaN,132729000000,144427000000,112010000000,1
1,2024-09-30,USD,180683000000,391035000000,210352000000,210352000000,123216000000,26097000000,31370000000,57467000000,NaN,0.0,0.000000e+00,0.000000e+00,NaN,NaN,NaN,11445000000,123485000000,29749000000,NaN,9.373600e+10,NaN,123216000000,134661000000,93736000000,1
2,2023-09-30,USD,169148000000,383285000000,214137000000,214137000000,114301000000,24932000000,29915000000,54847000000,NaN,-183000000.0,3.750000e+09,3.933000e+09,NaN,-382000000.0,NaN,11519000000,113736000000,16741000000,NaN,9.699500e+10,NaN,114301000000,125820000000,96995000000,1
3,2022-09-30,USD,170782000000,394328000000,223546000000,223546000000,119437000000,25094000000,26251000000,51573000000,NaN,-106000000.0,2.825000e+09,2.931000e+09,NaN,334000000.0,NaN,11104000000,119103000000,19300000000,NaN,9.980300e+10,NaN,119437000000,130541000000,99803000000,1
4,2021-09-30,USD,152836000000,365817000000,212981000000,212981000000,108949000000,21973000000,21914000000,43887000000,NaN,198000000.0,2.843000e+09,2.645000e+09,NaN,258000000.0,NaN,11284000000,109207000000,14527000000,NaN,9.468000e+10,NaN,111852000000,123136000000,94680000000,1


In [51]:
df2 = pd.read_csv('./stocks/MSFT/INCOME_STATEMENT.csv')
df2["stock_id"] = 2
df2.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,stock_id
0,2025-06-30,USD,193893000000,281724000000,87831000000,87831000000,128528000000,7223000000,32488000000,65365000000,NaN,2.620000e+08,7.670000e+08,2385000000,NaN,NaN,NaN,34153000000,123627000000,21795000000,NaN,1.018320e+11,NaN,126012000000,160165000000,101832000000,2
1,2024-06-30,USD,171008000000,245122000000,74114000000,74114000000,109433000000,7609000000,29510000000,61575000000,NaN,2.220000e+08,3.157000e+09,2935000000,NaN,NaN,NaN,22287000000,107787000000,19651000000,NaN,8.813600e+10,NaN,110722000000,133009000000,88136000000,2
2,2023-06-30,USD,146052000000,211915000000,65863000000,65863000000,88523000000,7575000000,27195000000,57529000000,NaN,1.026000e+09,1.041000e+09,1968000000,NaN,NaN,NaN,13861000000,89311000000,16950000000,NaN,7.236100e+10,NaN,91279000000,105140000000,72361000000,2
3,2022-06-30,USD,135620000000,198270000000,62650000000,62650000000,83383000000,5900000000,24512000000,52237000000,NaN,3.100000e+07,2.094000e+09,2063000000,NaN,3.330000e+08,NaN,14460000000,83716000000,10978000000,NaN,7.273800e+10,NaN,85779000000,100239000000,72738000000,2
4,2021-06-30,USD,115856000000,168088000000,52232000000,52232000000,69916000000,5107000000,20716000000,45940000000,NaN,-2.150000e+08,2.131000e+09,2346000000,NaN,1.186000e+09,NaN,11686000000,71102000000,9831000000,NaN,6.127100e+10,NaN,73448000000,85134000000,61271000000,2


In [50]:
df3 = pd.read_csv('./stocks/TSLA/INCOME_STATEMENT.csv')
df3["stock_id"] = 3
df3.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,stock_id
0,2025-12-31,USD,17094000000,9.482700e+10,77733000000,77733000000,4355000000,5.834000e+09,6411000000,12739000000,NaN,1.342000e+09,1.680000e+09,338000000,NaN,NaN,NaN,6148000000,5278000000,1423000000,NaN,3.855000e+09,NaN,5616000000,11764000000,3794000000,3
1,2024-12-31,USD,17450000000,9.769000e+10,80240000000,80240000000,7076000000,5.150000e+09,4540000000,10374000000,NaN,1.219000e+09,1.569000e+09,350000000,NaN,NaN,NaN,5368000000,8990000000,1837000000,NaN,7.153000e+09,NaN,9340000000,14708000000,7130000000,3
2,2023-12-31,USD,17660000000,9.677300e+10,79113000000,79113000000,8891000000,4.800000e+09,3969000000,8769000000,NaN,9.100000e+08,1.066000e+09,156000000,NaN,NaN,NaN,4667000000,9973000000,-5001000000,NaN,1.497400e+10,NaN,10129000000,14796000000,14974000000,3
3,2022-12-31,USD,20853000000,8.146200e+10,60609000000,60609000000,13656000000,3.946000e+09,3075000000,7021000000,NaN,1.060000e+08,4.100000e+07,191000000,NaN,NaN,NaN,3543000000,13719000000,1132000000,NaN,1.258700e+10,NaN,14290000000,17833000000,12587000000,3
4,2021-12-31,USD,13606000000,5.382300e+10,40217000000,40217000000,6687000000,4.517000e+09,2593000000,7110000000,NaN,-3.150000e+08,5.600000e+07,371000000,NaN,191000000.0,NaN,2911000000,6343000000,699000000,NaN,5.644000e+09,NaN,6687000000,9598000000,5644000000,3


In [53]:
df = pd.concat([df1,df2,df3],ignore_index=True)
#df.drop("Unnamed: 0",axis=1,inplace=True)
df.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,stock_id
0,2025-09-30,USD,195201000000,4.161610e+11,220960000000,220960000000,133050000000,8.077000e+09,34550000000,62151000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11698000000,132729000000,20719000000,NaN,1.120100e+11,NaN,132729000000,144427000000,112010000000,1
1,2024-09-30,USD,180683000000,3.910350e+11,210352000000,210352000000,123216000000,2.609700e+10,31370000000,57467000000,NaN,0.0,0.000000e+00,0.000000e+00,NaN,NaN,NaN,11445000000,123485000000,29749000000,NaN,9.373600e+10,NaN,123216000000,134661000000,93736000000,1
2,2023-09-30,USD,169148000000,3.832850e+11,214137000000,214137000000,114301000000,2.493200e+10,29915000000,54847000000,NaN,-183000000.0,3.750000e+09,3.933000e+09,NaN,-382000000.0,NaN,11519000000,113736000000,16741000000,NaN,9.699500e+10,NaN,114301000000,125820000000,96995000000,1
3,2022-09-30,USD,170782000000,3.943280e+11,223546000000,223546000000,119437000000,2.509400e+10,26251000000,51573000000,NaN,-106000000.0,2.825000e+09,2.931000e+09,NaN,334000000.0,NaN,11104000000,119103000000,19300000000,NaN,9.980300e+10,NaN,119437000000,130541000000,99803000000,1
4,2021-09-30,USD,152836000000,3.658170e+11,212981000000,212981000000,108949000000,2.197300e+10,21914000000,43887000000,NaN,198000000.0,2.843000e+09,2.645000e+09,NaN,258000000.0,NaN,11284000000,109207000000,14527000000,NaN,9.468000e+10,NaN,111852000000,123136000000,94680000000,1


In [54]:
df.columns

Index(['fiscalDateEnding', 'reportedCurrency', 'grossProfit', 'totalRevenue',
       'costOfRevenue', 'costofGoodsAndServicesSold', 'operatingIncome',
       'sellingGeneralAndAdministrative', 'researchAndDevelopment',
       'operatingExpenses', 'investmentIncomeNet', 'netInterestIncome',
       'interestIncome', 'interestExpense', 'nonInterestIncome',
       'otherNonOperatingIncome', 'depreciation',
       'depreciationAndAmortization', 'incomeBeforeTax', 'incomeTaxExpense',
       'interestAndDebtExpense', 'netIncomeFromContinuingOperations',
       'comprehensiveIncomeNetOfTax', 'ebit', 'ebitda', 'netIncome',
       'stock_id'],
      dtype='str')

In [55]:
column_mapping = {
    'fiscalDateEnding': 'fiscal_date_ending',
    'reportedCurrency': 'reported_currency',
    'grossProfit': 'gross_profit',
    'totalRevenue': 'total_revenue',
    'costOfRevenue': 'cost_of_revenue',
    'costofGoodsAndServicesSold': 'cost_of_goods_and_services_sold',
    'operatingIncome': 'operating_income',
    'sellingGeneralAndAdministrative': 'selling_general_and_administrative',
    'researchAndDevelopment': 'research_and_development',
    'operatingExpenses': 'operating_expenses',
    'investmentIncomeNet': 'investment_income_net',
    'netInterestIncome': 'net_interest_income',
    'interestIncome': 'interest_income',
    'interestExpense': 'interest_expense',
    'nonInterestIncome': 'non_interest_income',
    'otherNonOperatingIncome': 'other_non_operating_income',
    'depreciation': 'depreciation',
    'depreciationAndAmortization': 'depreciation_and_amortization',
    'incomeBeforeTax': 'income_before_tax',
    'incomeTaxExpense': 'income_tax_expense',
    'interestAndDebtExpense': 'interest_and_debt_expense',
    'netIncomeFromContinuingOperations': 'net_income_from_continuing_operations',
    'comprehensiveIncomeNetOfTax': 'comprehensive_income_net_of_tax',
    'ebit': 'ebit',
    'ebitda': 'ebitda',
    'netIncome': 'net_income',
    'stock_id': 'stock_id'
}


In [56]:
df.rename(columns=column_mapping,inplace=True)
df.head()

,fiscal_date_ending,reported_currency,gross_profit,total_revenue,cost_of_revenue,cost_of_goods_and_services_sold,operating_income,selling_general_and_administrative,research_and_development,operating_expenses,investment_income_net,net_interest_income,interest_income,interest_expense,non_interest_income,other_non_operating_income,depreciation,depreciation_and_amortization,income_before_tax,income_tax_expense,interest_and_debt_expense,net_income_from_continuing_operations,comprehensive_income_net_of_tax,ebit,ebitda,net_income,stock_id
0,2025-09-30,USD,195201000000,4.161610e+11,220960000000,220960000000,133050000000,8.077000e+09,34550000000,62151000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11698000000,132729000000,20719000000,NaN,1.120100e+11,NaN,132729000000,144427000000,112010000000,1
1,2024-09-30,USD,180683000000,3.910350e+11,210352000000,210352000000,123216000000,2.609700e+10,31370000000,57467000000,NaN,0.0,0.000000e+00,0.000000e+00,NaN,NaN,NaN,11445000000,123485000000,29749000000,NaN,9.373600e+10,NaN,123216000000,134661000000,93736000000,1
2,2023-09-30,USD,169148000000,3.832850e+11,214137000000,214137000000,114301000000,2.493200e+10,29915000000,54847000000,NaN,-183000000.0,3.750000e+09,3.933000e+09,NaN,-382000000.0,NaN,11519000000,113736000000,16741000000,NaN,9.699500e+10,NaN,114301000000,125820000000,96995000000,1
3,2022-09-30,USD,170782000000,3.943280e+11,223546000000,223546000000,119437000000,2.509400e+10,26251000000,51573000000,NaN,-106000000.0,2.825000e+09,2.931000e+09,NaN,334000000.0,NaN,11104000000,119103000000,19300000000,NaN,9.980300e+10,NaN,119437000000,130541000000,99803000000,1
4,2021-09-30,USD,152836000000,3.658170e+11,212981000000,212981000000,108949000000,2.197300e+10,21914000000,43887000000,NaN,198000000.0,2.843000e+09,2.645000e+09,NaN,258000000.0,NaN,11284000000,109207000000,14527000000,NaN,9.468000e+10,NaN,111852000000,123136000000,94680000000,1


In [57]:
df.to_csv('../stock_db/data/income_statement.csv',index=False)

### Balance sheet

In [4]:
df1 = pd.read_csv('./stocks/AAPL/BALANCE_SHEET.csv')
df1["stock_id"] = 1
df1.head()

,fiscalDateEnding,reportedCurrency,totalAssets,totalCurrentAssets,cashAndCashEquivalentsAtCarryingValue,cashAndShortTermInvestments,inventory,currentNetReceivables,totalNonCurrentAssets,propertyPlantEquipment,accumulatedDepreciationAmortizationPPE,intangibleAssets,intangibleAssetsExcludingGoodwill,goodwill,investments,longTermInvestments,shortTermInvestments,otherCurrentAssets,otherNonCurrentAssets,totalLiabilities,totalCurrentLiabilities,currentAccountsPayable,deferredRevenue,currentDebt,shortTermDebt,totalNonCurrentLiabilities,capitalLeaseObligations,longTermDebt,currentLongTermDebt,longTermDebtNoncurrent,shortLongTermDebtTotal,otherCurrentLiabilities,otherNonCurrentLiabilities,totalShareholderEquity,treasuryStock,retainedEarnings,commonStock,commonStockSharesOutstanding,stock_id
0,2025-09-30,USD,359241000000,147957000000,35934000000,35934000000,5718000000,72957000000,211284000000,61039000000,NaN,NaN,NaN,NaN,NaN,7.772300e+10,18763000000,14585000000,NaN,285508000000,165631000000,69860000000,NaN,NaN,22446000000,119877000000,NaN,7.832800e+10,2.032900e+10,NaN,1.123770e+11,64270000000,4.154900e+10,73733000000,NaN,-14264000000,93568000000,15004697000,1
1,2024-09-30,USD,364980000000,152987000000,29943000000,29943000000,7286000000,66243000000,211993000000,55914000000,NaN,NaN,NaN,NaN,NaN,9.147900e+10,35228000000,14287000000,NaN,308030000000,176392000000,68960000000,NaN,NaN,22511000000,131638000000,NaN,8.575000e+10,2.087900e+10,NaN,1.190590e+11,50071000000,3.663400e+10,56950000000,NaN,-19154000000,83276000000,15408095000,1
2,2023-09-30,USD,352583000000,143566000000,29965000000,29965000000,6331000000,29508000000,209017000000,43715000000,NaN,NaN,NaN,NaN,NaN,1.005440e+11,31590000000,14695000000,NaN,290437000000,145308000000,62611000000,NaN,NaN,15807000000,145129000000,1.284200e+10,9.528100e+10,1.580700e+10,NaN,1.119470e+11,58829000000,4.984800e+10,62146000000,NaN,-214000000,73812000000,15812547000,1
3,2022-09-30,USD,352755000000,135405000000,23646000000,23646000000,4946000000,28184000000,217350000000,42117000000,7.234000e+10,NaN,NaN,NaN,NaN,1.208050e+11,24658000000,21223000000,NaN,302083000000,153982000000,64115000000,NaN,NaN,21110000000,148101000000,1.241100e+10,9.895900e+10,2.111000e+10,NaN,1.208810e+11,60845000000,4.914200e+10,50672000000,NaN,-3068000000,64849000000,16325819000,1
4,2021-09-30,USD,351002000000,134836000000,34940000000,34940000000,6580000000,26278000000,216166000000,39440000000,7.028300e+10,NaN,NaN,NaN,NaN,1.278770e+11,27699000000,14111000000,NaN,287912000000,125481000000,54763000000,NaN,NaN,15613000000,162431000000,1.180300e+10,1.091060e+11,1.561300e+10,NaN,1.247190e+11,47493000000,2.863600e+10,63090000000,NaN,5562000000,57365000000,16864919000,1


In [5]:
df2 = pd.read_csv('./stocks/MSFT/BALANCE_SHEET.csv')
df2["stock_id"] = 2
df2.head()

,fiscalDateEnding,reportedCurrency,totalAssets,totalCurrentAssets,cashAndCashEquivalentsAtCarryingValue,cashAndShortTermInvestments,inventory,currentNetReceivables,totalNonCurrentAssets,propertyPlantEquipment,accumulatedDepreciationAmortizationPPE,intangibleAssets,intangibleAssetsExcludingGoodwill,goodwill,investments,longTermInvestments,shortTermInvestments,otherCurrentAssets,otherNonCurrentAssets,totalLiabilities,totalCurrentLiabilities,currentAccountsPayable,deferredRevenue,currentDebt,shortTermDebt,totalNonCurrentLiabilities,capitalLeaseObligations,longTermDebt,currentLongTermDebt,longTermDebtNoncurrent,shortLongTermDebtTotal,otherCurrentLiabilities,otherNonCurrentLiabilities,totalShareholderEquity,treasuryStock,retainedEarnings,commonStock,commonStockSharesOutstanding,stock_id
0,2025-06-30,USD,619003000000,191131000000,30242000000,30242000000,938000000,69905000000,427872000000,229789000000,NaN,22604000000,22604000000,119509000000,NaN,1.513300e+10,64313000000,25733000000,NaN,275524000000,141218000000,27724000000,NaN,NaN,11595000000,134306000000,1.743700e+10,4.015200e+10,2.999000e+09,NaN,1.121840e+11,37344000000,4.518600e+10,343479000000,NaN,237731000000,109095000000,7465000000,2
1,2024-06-30,USD,512163000000,159734000000,18315000000,18315000000,1246000000,56924000000,352429000000,154552000000,NaN,27597000000,27597000000,119220000000,NaN,1.460000e+10,57216000000,26033000000,NaN,243686000000,125286000000,21996000000,NaN,NaN,14871000000,118400000000,1.549700e+10,4.268800e+10,8.942000e+09,NaN,9.785200e+10,25820000000,2.706400e+10,268477000000,NaN,173144000000,100923000000,7469000000,2
2,2023-06-30,USD,411976000000,184257000000,34704000000,34704000000,2500000000,48688000000,227719000000,109987000000,NaN,9366000000,9366000000,67886000000,NaN,9.879000e+09,76558000000,21807000000,NaN,205753000000,104149000000,18095000000,NaN,NaN,5247000000,101604000000,1.272800e+10,4.199000e+10,5.247000e+09,NaN,5.996500e+10,29906000000,1.798100e+10,206223000000,NaN,118848000000,93718000000,7472000000,2
3,2022-06-30,USD,364840000000,169684000000,13931000000,13931000000,3742000000,44261000000,195156000000,87546000000,NaN,11298000000,11298000000,67524000000,NaN,6.891000e+09,90826000000,16924000000,NaN,198298000000,95082000000,19000000000,NaN,NaN,2749000000,103216000000,1.148900e+10,4.703200e+10,2.749000e+09,NaN,6.127000e+10,27795000000,1.552600e+10,166542000000,NaN,84281000000,86939000000,7540000000,2
4,2021-06-30,USD,333779000000,184406000000,14224000000,14224000000,2636000000,38043000000,149373000000,70803000000,NaN,7800000000,7800000000,49711000000,NaN,5.984000e+09,116110000000,13393000000,NaN,191791000000,88657000000,15163000000,NaN,NaN,8072000000,103134000000,9.629000e+09,5.007400e+10,8.072000e+09,NaN,6.777500e+10,23897000000,1.342700e+10,141988000000,NaN,57055000000,83111000000,7608000000,2


In [6]:
df3 = pd.read_csv('./stocks/TSLA/BALANCE_SHEET.csv')
df3["stock_id"] = 3
df3.head()

,fiscalDateEnding,reportedCurrency,totalAssets,totalCurrentAssets,cashAndCashEquivalentsAtCarryingValue,cashAndShortTermInvestments,inventory,currentNetReceivables,totalNonCurrentAssets,propertyPlantEquipment,accumulatedDepreciationAmortizationPPE,intangibleAssets,intangibleAssetsExcludingGoodwill,goodwill,investments,longTermInvestments,shortTermInvestments,otherCurrentAssets,otherNonCurrentAssets,totalLiabilities,totalCurrentLiabilities,currentAccountsPayable,deferredRevenue,currentDebt,shortTermDebt,totalNonCurrentLiabilities,capitalLeaseObligations,longTermDebt,currentLongTermDebt,longTermDebtNoncurrent,shortLongTermDebtTotal,otherCurrentLiabilities,otherNonCurrentLiabilities,totalShareholderEquity,treasuryStock,retainedEarnings,commonStock,commonStockSharesOutstanding,stock_id
0,2025-12-31,USD,1.378060e+11,6.864200e+10,1.651300e+10,1.651300e+10,1.239200e+10,4.576000e+09,6.916400e+10,5.618600e+10,NaN,1.350000e+08,1.350000e+08,257000000.0,NaN,NaN,2.754600e+10,7.615000e+09,NaN,5.494100e+10,3.171400e+10,1.337100e+10,NaN,NaN,1.640000e+09,2.322700e+10,6.566000e+09,6.584000e+09,1.569000e+09,NaN,8.376000e+09,1.327900e+10,1.339000e+09,8.213700e+10,NaN,3.900300e+10,3000000.0,3528000000,3
1,2024-12-31,USD,1.220700e+11,5.836000e+10,1.613900e+10,1.613900e+10,1.201700e+10,4.418000e+09,6.371600e+10,5.150100e+10,NaN,1.226000e+09,1.226000e+09,244000000.0,NaN,NaN,2.042400e+10,5.362000e+09,NaN,4.839000e+10,2.882100e+10,1.247400e+10,NaN,NaN,3.263000e+09,1.956900e+10,5.745000e+09,5.535000e+09,2.343000e+09,NaN,1.362300e+10,7.556000e+09,1.093000e+09,7.291300e+10,NaN,3.520900e+10,3000000.0,3498000000,3
2,2023-12-31,USD,1.066180e+11,4.961600e+10,1.639800e+10,1.639800e+10,1.362600e+10,3.508000e+09,5.700300e+10,4.512300e+10,NaN,3.620000e+08,3.620000e+08,253000000.0,NaN,NaN,1.269600e+10,3.388000e+09,NaN,4.300900e+10,2.874800e+10,1.443100e+10,NaN,NaN,3.045000e+09,1.426100e+10,4.916000e+09,2.682000e+09,1.975000e+09,NaN,9.573000e+09,6.328000e+09,8.760000e+08,6.263400e+10,NaN,2.788200e+10,3000000.0,3482750000,3
3,2022-12-31,USD,8.233800e+10,4.091700e+10,1.625300e+10,1.625300e+10,1.283900e+10,2.952000e+09,4.142100e+10,3.663500e+10,NaN,3.990000e+08,3.990000e+08,194000000.0,NaN,NaN,5.932000e+09,2.941000e+09,NaN,3.644000e+10,2.670900e+10,1.525500e+10,NaN,NaN,1.987000e+09,9.731000e+09,3.703000e+09,1.029000e+09,1.016000e+09,NaN,5.748000e+09,5.422000e+09,5.530000e+08,4.470400e+10,NaN,1.288500e+10,3000000.0,3475000000,3
4,2021-12-31,USD,6.213100e+10,2.710000e+10,1.757600e+10,1.757600e+10,5.757000e+09,1.913000e+09,3.503100e+10,3.117600e+10,NaN,1.517000e+09,1.517000e+09,200000000.0,NaN,NaN,1.310000e+08,1.723000e+09,NaN,3.111600e+10,1.970500e+10,1.002500e+10,NaN,NaN,1.589000e+09,1.141100e+10,3.531000e+09,4.254000e+09,1.088000e+09,NaN,8.873000e+09,5.719000e+09,1.084300e+10,3.101500e+10,NaN,3.310000e+08,1000000.0,3100522833,3


In [7]:
df = pd.concat([df1,df2,df3],ignore_index=True)
#df.drop("Unnamed: 0",axis=1,inplace=True)
df.head()

,fiscalDateEnding,reportedCurrency,totalAssets,totalCurrentAssets,cashAndCashEquivalentsAtCarryingValue,cashAndShortTermInvestments,inventory,currentNetReceivables,totalNonCurrentAssets,propertyPlantEquipment,accumulatedDepreciationAmortizationPPE,intangibleAssets,intangibleAssetsExcludingGoodwill,goodwill,investments,longTermInvestments,shortTermInvestments,otherCurrentAssets,otherNonCurrentAssets,totalLiabilities,totalCurrentLiabilities,currentAccountsPayable,deferredRevenue,currentDebt,shortTermDebt,totalNonCurrentLiabilities,capitalLeaseObligations,longTermDebt,currentLongTermDebt,longTermDebtNoncurrent,shortLongTermDebtTotal,otherCurrentLiabilities,otherNonCurrentLiabilities,totalShareholderEquity,treasuryStock,retainedEarnings,commonStock,commonStockSharesOutstanding,stock_id
0,2025-09-30,USD,3.592410e+11,1.479570e+11,3.593400e+10,3.593400e+10,5.718000e+09,7.295700e+10,2.112840e+11,6.103900e+10,NaN,NaN,NaN,NaN,NaN,7.772300e+10,1.876300e+10,1.458500e+10,NaN,2.855080e+11,1.656310e+11,6.986000e+10,NaN,NaN,2.244600e+10,1.198770e+11,NaN,7.832800e+10,2.032900e+10,NaN,1.123770e+11,6.427000e+10,4.154900e+10,7.373300e+10,NaN,-1.426400e+10,9.356800e+10,15004697000,1
1,2024-09-30,USD,3.649800e+11,1.529870e+11,2.994300e+10,2.994300e+10,7.286000e+09,6.624300e+10,2.119930e+11,5.591400e+10,NaN,NaN,NaN,NaN,NaN,9.147900e+10,3.522800e+10,1.428700e+10,NaN,3.080300e+11,1.763920e+11,6.896000e+10,NaN,NaN,2.251100e+10,1.316380e+11,NaN,8.575000e+10,2.087900e+10,NaN,1.190590e+11,5.007100e+10,3.663400e+10,5.695000e+10,NaN,-1.915400e+10,8.327600e+10,15408095000,1
2,2023-09-30,USD,3.525830e+11,1.435660e+11,2.996500e+10,2.996500e+10,6.331000e+09,2.950800e+10,2.090170e+11,4.371500e+10,NaN,NaN,NaN,NaN,NaN,1.005440e+11,3.159000e+10,1.469500e+10,NaN,2.904370e+11,1.453080e+11,6.261100e+10,NaN,NaN,1.580700e+10,1.451290e+11,1.284200e+10,9.528100e+10,1.580700e+10,NaN,1.119470e+11,5.882900e+10,4.984800e+10,6.214600e+10,NaN,-2.140000e+08,7.381200e+10,15812547000,1
3,2022-09-30,USD,3.527550e+11,1.354050e+11,2.364600e+10,2.364600e+10,4.946000e+09,2.818400e+10,2.173500e+11,4.211700e+10,7.234000e+10,NaN,NaN,NaN,NaN,1.208050e+11,2.465800e+10,2.122300e+10,NaN,3.020830e+11,1.539820e+11,6.411500e+10,NaN,NaN,2.111000e+10,1.481010e+11,1.241100e+10,9.895900e+10,2.111000e+10,NaN,1.208810e+11,6.084500e+10,4.914200e+10,5.067200e+10,NaN,-3.068000e+09,6.484900e+10,16325819000,1
4,2021-09-30,USD,3.510020e+11,1.348360e+11,3.494000e+10,3.494000e+10,6.580000e+09,2.627800e+10,2.161660e+11,3.944000e+10,7.028300e+10,NaN,NaN,NaN,NaN,1.278770e+11,2.769900e+10,1.411100e+10,NaN,2.879120e+11,1.254810e+11,5.476300e+10,NaN,NaN,1.561300e+10,1.624310e+11,1.180300e+10,1.091060e+11,1.561300e+10,NaN,1.247190e+11,4.749300e+10,2.863600e+10,6.309000e+10,NaN,5.562000e+09,5.736500e+10,16864919000,1


In [62]:
df.columns

Index(['fiscalDateEnding', 'reportedCurrency', 'totalAssets',
       'totalCurrentAssets', 'cashAndCashEquivalentsAtCarryingValue',
       'cashAndShortTermInvestments', 'inventory', 'currentNetReceivables',
       'totalNonCurrentAssets', 'propertyPlantEquipment',
       'accumulatedDepreciationAmortizationPPE', 'intangibleAssets',
       'intangibleAssetsExcludingGoodwill', 'goodwill', 'investments',
       'longTermInvestments', 'shortTermInvestments', 'otherCurrentAssets',
       'otherNonCurrentAssets', 'totalLiabilities', 'totalCurrentLiabilities',
       'currentAccountsPayable', 'deferredRevenue', 'currentDebt',
       'shortTermDebt', 'totalNonCurrentLiabilities',
       'capitalLeaseObligations', 'longTermDebt', 'currentLongTermDebt',
       'longTermDebtNoncurrent', 'shortLongTermDebtTotal',
       'otherCurrentLiabilities', 'otherNonCurrentLiabilities',
       'totalShareholderEquity', 'treasuryStock', 'retainedEarnings',
       'commonStock', 'commonStockSharesOutstanding'

In [8]:
balance_sheet_mapping = {
    'fiscalDateEnding': 'fiscal_date_ending',
    'reportedCurrency': 'reported_currency',
    'totalAssets': 'total_assets',
    'totalCurrentAssets': 'total_current_assets',
    'cashAndCashEquivalentsAtCarryingValue': 'cash_and_cash_equivalents_at_carrying_value',
    'cashAndShortTermInvestments': 'cash_and_short_term_investments',
    'inventory': 'inventory',
    'currentNetReceivables': 'current_net_receivables',
    'totalNonCurrentAssets': 'total_non_current_assets',
    'propertyPlantEquipment': 'property_plant_equipment',
    'accumulatedDepreciationAmortizationPPE': 'accumulated_depreciation_amortization_ppe',
    'intangibleAssets': 'intangible_assets',
    'intangibleAssetsExcludingGoodwill': 'intangible_assets_excluding_goodwill',
    'goodwill': 'goodwill',
    'investments': 'investments',
    'longTermInvestments': 'long_term_investments',
    'shortTermInvestments': 'short_term_investments',
    'otherCurrentAssets': 'other_current_assets',
    'otherNonCurrentAssets': 'other_non_current_assets',
    'totalLiabilities': 'total_liabilities',
    'totalCurrentLiabilities': 'total_current_liabilities',
    'currentAccountsPayable': 'current_accounts_payable',
    'deferredRevenue': 'deferred_revenue',
    'currentDebt': 'current_debt',
    'shortTermDebt': 'short_term_debt',
    'totalNonCurrentLiabilities': 'total_non_current_liabilities',
    'capitalLeaseObligations': 'capital_lease_obligations',
    'longTermDebt': 'long_term_debt',
    'currentLongTermDebt': 'current_long_term_debt',
    'longTermDebtNoncurrent': 'long_term_debt_noncurrent',
    'shortLongTermDebtTotal': 'short_long_term_debt_total',
    'otherCurrentLiabilities': 'other_current_liabilities',
    'otherNonCurrentLiabilities': 'other_non_current_liabilities',
    'totalShareholderEquity': 'total_shareholder_equity',
    'treasuryStock': 'treasury_stock',
    'retainedEarnings': 'retained_earnings',
    'commonStock': 'common_stock',
    'commonStockSharesOutstanding': 'common_stock_shares_outstanding',
    'stock_id': 'stock_id'
}

# Apply to your dataframe
df.rename(columns=balance_sheet_mapping, inplace=True)


In [9]:
df.to_csv('../stock_db/data/balance_sheet.csv',index=False)

### Cash flow statement

In [65]:
df1 = pd.read_csv('./stocks/AAPL/CASH_FLOW.csv')
df1["stock_id"] = 1
df1.head()

,fiscalDateEnding,reportedCurrency,operatingCashflow,paymentsForOperatingActivities,proceedsFromOperatingActivities,changeInOperatingLiabilities,changeInOperatingAssets,depreciationDepletionAndAmortization,capitalExpenditures,changeInReceivables,changeInInventory,profitLoss,cashflowFromInvestment,cashflowFromFinancing,proceedsFromRepaymentsOfShortTermDebt,paymentsForRepurchaseOfCommonStock,paymentsForRepurchaseOfEquity,paymentsForRepurchaseOfPreferredStock,dividendPayout,dividendPayoutCommonStock,dividendPayoutPreferredStock,proceedsFromIssuanceOfCommonStock,proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet,proceedsFromIssuanceOfPreferredStock,proceedsFromRepurchaseOfEquity,proceedsFromSaleOfTreasuryStock,stockBasedCompensation,changeInCashAndCashEquivalents,changeInExchangeRate,netIncome,stock_id
0,2025-09-30,USD,111482000000,NaN,NaN,NaN,NaN,11698000000,12715000000,NaN,1400000000,NaN,15195000000,-120686000000,NaN,NaN,NaN,NaN,1.542100e+10,1.542100e+10,NaN,NaN,NaN,NaN,-90711000000,NaN,12863000000,NaN,NaN,112010000000,1
1,2024-09-30,USD,118254000000,NaN,NaN,NaN,NaN,11445000000,9447000000,NaN,-1046000000,NaN,2935000000,-121983000000,NaN,NaN,NaN,NaN,1.523400e+10,1.523400e+10,NaN,NaN,NaN,NaN,-94949000000,NaN,11688000000,NaN,NaN,93736000000,1
2,2023-09-30,USD,110543000000,NaN,NaN,NaN,NaN,11519000000,10959000000,-4.170000e+08,-1618000000,NaN,3705000000,-108488000000,NaN,NaN,NaN,NaN,1.502500e+10,1.502500e+10,NaN,NaN,NaN,NaN,-77550000000,NaN,10833000000,5.760000e+09,NaN,96995000000,1
3,2022-09-30,USD,122151000000,NaN,NaN,NaN,NaN,11104000000,10708000000,-9.343000e+09,1484000000,NaN,-22354000000,-110749000000,NaN,NaN,NaN,NaN,1.484100e+10,1.484100e+10,NaN,NaN,NaN,NaN,-89402000000,NaN,9038000000,-1.095200e+10,NaN,99803000000,1
4,2021-09-30,USD,104038000000,NaN,NaN,NaN,NaN,11284000000,11085000000,-1.402800e+10,-2642000000,NaN,-14545000000,-93353000000,NaN,NaN,NaN,NaN,1.446700e+10,1.446700e+10,NaN,NaN,NaN,NaN,-85971000000,NaN,7906000000,-3.860000e+09,NaN,94680000000,1


In [66]:
df2 = pd.read_csv('./stocks/MSFT/CASH_FLOW.csv')
df2["stock_id"] = 2
df2.head()

,fiscalDateEnding,reportedCurrency,operatingCashflow,paymentsForOperatingActivities,proceedsFromOperatingActivities,changeInOperatingLiabilities,changeInOperatingAssets,depreciationDepletionAndAmortization,capitalExpenditures,changeInReceivables,changeInInventory,profitLoss,cashflowFromInvestment,cashflowFromFinancing,proceedsFromRepaymentsOfShortTermDebt,paymentsForRepurchaseOfCommonStock,paymentsForRepurchaseOfEquity,paymentsForRepurchaseOfPreferredStock,dividendPayout,dividendPayoutCommonStock,dividendPayoutPreferredStock,proceedsFromIssuanceOfCommonStock,proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet,proceedsFromIssuanceOfPreferredStock,proceedsFromRepurchaseOfEquity,proceedsFromSaleOfTreasuryStock,stockBasedCompensation,changeInCashAndCashEquivalents,changeInExchangeRate,netIncome,stock_id
0,2025-06-30,USD,136162000000,NaN,NaN,NaN,NaN,34153000000,64551000000,NaN,3.090000e+08,NaN,-72599000000,-51699000000,NaN,NaN,NaN,NaN,24082000000,24082000000,NaN,NaN,NaN,NaN,-18420000000,NaN,11974000000,NaN,NaN,101832000000,2
1,2024-06-30,USD,118548000000,NaN,NaN,NaN,NaN,22287000000,44477000000,NaN,1.284000e+09,NaN,-96970000000,-37757000000,NaN,NaN,NaN,NaN,21771000000,21771000000,NaN,NaN,NaN,NaN,-17254000000,NaN,10734000000,NaN,NaN,88136000000,2
2,2023-06-30,USD,87582000000,NaN,NaN,NaN,NaN,13861000000,28107000000,NaN,1.242000e+09,NaN,-22680000000,-43935000000,NaN,NaN,NaN,NaN,19800000000,19800000000,NaN,NaN,NaN,NaN,-22245000000,NaN,9611000000,NaN,NaN,72361000000,2
3,2022-06-30,USD,89035000000,NaN,NaN,NaN,NaN,14460000000,23886000000,-6.834000e+09,-1.123000e+09,NaN,-30311000000,-58876000000,NaN,NaN,NaN,NaN,18135000000,18135000000,NaN,NaN,NaN,NaN,-32696000000,NaN,7502000000,-152000000.0,NaN,72738000000,2
4,2021-06-30,USD,76740000000,NaN,NaN,NaN,NaN,11686000000,20622000000,-6.481000e+09,-7.370000e+08,NaN,-27577000000,-48486000000,NaN,NaN,NaN,NaN,16521000000,16521000000,NaN,NaN,NaN,NaN,-27385000000,NaN,6118000000,648000000.0,-29000000.0,61271000000,2


In [67]:
df3 = pd.read_csv('./stocks/TSLA/CASH_FLOW.csv')
df3["stock_id"] = 3
df3.head()

,fiscalDateEnding,reportedCurrency,operatingCashflow,paymentsForOperatingActivities,proceedsFromOperatingActivities,changeInOperatingLiabilities,changeInOperatingAssets,depreciationDepletionAndAmortization,capitalExpenditures,changeInReceivables,changeInInventory,profitLoss,cashflowFromInvestment,cashflowFromFinancing,proceedsFromRepaymentsOfShortTermDebt,paymentsForRepurchaseOfCommonStock,paymentsForRepurchaseOfEquity,paymentsForRepurchaseOfPreferredStock,dividendPayout,dividendPayoutCommonStock,dividendPayoutPreferredStock,proceedsFromIssuanceOfCommonStock,proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet,proceedsFromIssuanceOfPreferredStock,proceedsFromRepurchaseOfEquity,proceedsFromSaleOfTreasuryStock,stockBasedCompensation,changeInCashAndCashEquivalents,changeInExchangeRate,netIncome,stock_id
0,2025-12-31,USD,14747000000,NaN,NaN,NaN,NaN,6148000000,8527000000,NaN,-6.300000e+08,NaN,-15478000000,1139000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.186000e+09,NaN,2.825000e+09,NaN,NaN,3855000000,3
1,2024-12-31,USD,14923000000,NaN,NaN,NaN,NaN,5368000000,11342000000,NaN,9.370000e+08,NaN,-18787000000,3853000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.241000e+09,NaN,1.999000e+09,NaN,NaN,7130000000,3
2,2023-12-31,USD,13256000000,NaN,NaN,NaN,NaN,4667000000,8899000000,NaN,-1.195000e+09,NaN,-15584000000,2589000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.000000e+08,NaN,1.812000e+09,NaN,NaN,14999000000,3
3,2022-12-31,USD,14724000000,NaN,NaN,NaN,NaN,3543000000,7172000000,NaN,-6.465000e+09,NaN,-11973000000,-3527000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.410000e+08,NaN,1.560000e+09,NaN,NaN,12583000000,3
4,2021-12-31,USD,11497000000,NaN,NaN,NaN,NaN,2911000000,8014000000,130000000.0,-1.709000e+09,NaN,-7868000000,-5203000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.480000e+08,NaN,2.121000e+09,-1.574000e+09,NaN,5519000000,3


In [72]:
df = pd.concat([df1,df2,df3],ignore_index=True)
#df.drop("Unnamed: 0",axis=1,inplace=True)
df.head()

,fiscalDateEnding,reportedCurrency,operatingCashflow,paymentsForOperatingActivities,proceedsFromOperatingActivities,changeInOperatingLiabilities,changeInOperatingAssets,depreciationDepletionAndAmortization,capitalExpenditures,changeInReceivables,changeInInventory,profitLoss,cashflowFromInvestment,cashflowFromFinancing,proceedsFromRepaymentsOfShortTermDebt,paymentsForRepurchaseOfCommonStock,paymentsForRepurchaseOfEquity,paymentsForRepurchaseOfPreferredStock,dividendPayout,dividendPayoutCommonStock,dividendPayoutPreferredStock,proceedsFromIssuanceOfCommonStock,proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet,proceedsFromIssuanceOfPreferredStock,proceedsFromRepurchaseOfEquity,proceedsFromSaleOfTreasuryStock,stockBasedCompensation,changeInCashAndCashEquivalents,changeInExchangeRate,netIncome,stock_id
0,2025-09-30,USD,111482000000,NaN,NaN,NaN,NaN,11698000000,12715000000,NaN,1.400000e+09,NaN,15195000000,-120686000000,NaN,NaN,NaN,NaN,1.542100e+10,1.542100e+10,NaN,NaN,NaN,NaN,-9.071100e+10,NaN,1.286300e+10,NaN,NaN,112010000000,1
1,2024-09-30,USD,118254000000,NaN,NaN,NaN,NaN,11445000000,9447000000,NaN,-1.046000e+09,NaN,2935000000,-121983000000,NaN,NaN,NaN,NaN,1.523400e+10,1.523400e+10,NaN,NaN,NaN,NaN,-9.494900e+10,NaN,1.168800e+10,NaN,NaN,93736000000,1
2,2023-09-30,USD,110543000000,NaN,NaN,NaN,NaN,11519000000,10959000000,-4.170000e+08,-1.618000e+09,NaN,3705000000,-108488000000,NaN,NaN,NaN,NaN,1.502500e+10,1.502500e+10,NaN,NaN,NaN,NaN,-7.755000e+10,NaN,1.083300e+10,5.760000e+09,NaN,96995000000,1
3,2022-09-30,USD,122151000000,NaN,NaN,NaN,NaN,11104000000,10708000000,-9.343000e+09,1.484000e+09,NaN,-22354000000,-110749000000,NaN,NaN,NaN,NaN,1.484100e+10,1.484100e+10,NaN,NaN,NaN,NaN,-8.940200e+10,NaN,9.038000e+09,-1.095200e+10,NaN,99803000000,1
4,2021-09-30,USD,104038000000,NaN,NaN,NaN,NaN,11284000000,11085000000,-1.402800e+10,-2.642000e+09,NaN,-14545000000,-93353000000,NaN,NaN,NaN,NaN,1.446700e+10,1.446700e+10,NaN,NaN,NaN,NaN,-8.597100e+10,NaN,7.906000e+09,-3.860000e+09,NaN,94680000000,1


In [69]:
df.columns

Index(['fiscalDateEnding', 'reportedCurrency', 'operatingCashflow',
       'paymentsForOperatingActivities', 'proceedsFromOperatingActivities',
       'changeInOperatingLiabilities', 'changeInOperatingAssets',
       'depreciationDepletionAndAmortization', 'capitalExpenditures',
       'changeInReceivables', 'changeInInventory', 'profitLoss',
       'cashflowFromInvestment', 'cashflowFromFinancing',
       'proceedsFromRepaymentsOfShortTermDebt',
       'paymentsForRepurchaseOfCommonStock', 'paymentsForRepurchaseOfEquity',
       'paymentsForRepurchaseOfPreferredStock', 'dividendPayout',
       'dividendPayoutCommonStock', 'dividendPayoutPreferredStock',
       'proceedsFromIssuanceOfCommonStock',
       'proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet',
       'proceedsFromIssuanceOfPreferredStock',
       'proceedsFromRepurchaseOfEquity', 'proceedsFromSaleOfTreasuryStock',
       'stockBasedCompensation', 'changeInCashAndCashEquivalents',
       'changeInExchangeRate', 'net

In [73]:
cash_flow_mapping = {
    'fiscalDateEnding': 'fiscal_date_ending',
    'reportedCurrency': 'reported_currency',
    'operatingCashflow': 'operating_cashflow',
    'paymentsForOperatingActivities': 'payments_for_operating_activities',
    'proceedsFromOperatingActivities': 'proceeds_from_operating_activities',
    'changeInOperatingLiabilities': 'change_in_operating_liabilities',
    'changeInOperatingAssets': 'change_in_operating_assets',
    'depreciationDepletionAndAmortization': 'depreciation_depletion_and_amortization',
    'capitalExpenditures': 'capital_expenditures',
    'changeInReceivables': 'change_in_receivables',
    'changeInInventory': 'change_in_inventory',
    'profitLoss': 'profit_loss',
    'cashflowFromInvestment': 'cashflow_from_investment',
    'cashflowFromFinancing': 'cashflow_from_financing',
    'proceedsFromRepaymentsOfShortTermDebt': 'proceeds_from_repayments_of_short_term_debt',
    'paymentsForRepurchaseOfCommonStock': 'payments_for_repurchase_of_common_stock',
    'paymentsForRepurchaseOfEquity': 'payments_for_repurchase_of_equity',
    'paymentsForRepurchaseOfPreferredStock': 'payments_for_repurchase_of_preferred_stock',
    'dividendPayout': 'dividend_payout',
    'dividendPayoutCommonStock': 'dividend_payout_common_stock',
    'dividendPayoutPreferredStock': 'dividend_payout_preferred_stock',
    'proceedsFromIssuanceOfCommonStock': 'proceeds_from_issuance_of_common_stock',
    'proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet': 'proceeds_from_issuance_of_long_term_debt_and_capital_securities',
    'proceedsFromIssuanceOfPreferredStock': 'proceeds_from_issuance_of_preferred_stock',
    'proceedsFromRepurchaseOfEquity': 'proceeds_from_repurchase_of_equity',
    'proceedsFromSaleOfTreasuryStock': 'proceeds_from_sale_of_treasury_stock',
    'stockBasedCompensation': 'stock_based_compensation',
    'changeInCashAndCashEquivalents': 'change_in_cash_and_cash_equivalents',
    'changeInExchangeRate': 'change_in_exchange_rate',
    'netIncome': 'net_income',
    'stock_id': 'stock_id'
}

df.rename(columns = cash_flow_mapping,inplace=True)

In [74]:
df.to_csv('../stock_db/data/cash_flow.csv',index=False)

### Earnings estimates

In [76]:
df1 = pd.read_csv('./stocks/AAPL/EARNINGS_ESTIMATES.csv')
df1["stock_id"] = 1
df1.head()

,Unnamed: 0,date,horizon,eps_estimate_average,eps_estimate_high,eps_estimate_low,eps_estimate_analyst_count,eps_estimate_average_7_days_ago,eps_estimate_average_30_days_ago,eps_estimate_average_60_days_ago,eps_estimate_average_90_days_ago,eps_estimate_revision_up_trailing_7_days,eps_estimate_revision_down_trailing_7_days,eps_estimate_revision_up_trailing_30_days,eps_estimate_revision_down_trailing_30_days,revenue_estimate_average,revenue_estimate_high,revenue_estimate_low,revenue_estimate_analyst_count,stock_id
0,0,2027-09-30,fiscal year,9.3883,10.38,8.4300,40.0,9.3279,9.3600,9.3273,9.1317,2.0,NaN,3.0,6.0,5.006671e+11,5.501531e+11,4.738811e+11,41.0,1
1,1,2026-09-30,fiscal year,8.5227,8.97,7.9600,41.0,8.5065,8.5123,8.5070,8.2624,2.0,NaN,4.0,4.0,4.666955e+11,4.857853e+11,4.487370e+11,41.0,1
2,2,2026-06-30,fiscal quarter,1.7408,1.89,1.5857,30.0,1.7294,1.7351,1.7325,1.7054,1.0,NaN,3.0,2.0,1.024719e+11,1.093279e+11,9.810500e+10,29.0,1
3,3,2026-03-31,fiscal quarter,1.9463,2.16,1.5600,32.0,1.9373,1.9554,1.9529,1.8473,1.0,NaN,3.0,2.0,1.096898e+11,1.153689e+11,1.070750e+11,31.0,1
4,4,2025-12-31,fiscal quarter,2.6708,2.80,2.4500,33.0,2.6731,2.6672,2.6632,2.6509,3.0,NaN,5.0,1.0,1.385221e+11,1.424600e+11,1.366560e+11,32.0,1


In [77]:
df2 = pd.read_csv('./stocks/MSFT/EARNINGS_ESTIMATES.csv')
df2["stock_id"] = 2
df2.head()

,Unnamed: 0,date,horizon,eps_estimate_average,eps_estimate_high,eps_estimate_low,eps_estimate_analyst_count,eps_estimate_average_7_days_ago,eps_estimate_average_30_days_ago,eps_estimate_average_60_days_ago,eps_estimate_average_90_days_ago,eps_estimate_revision_up_trailing_7_days,eps_estimate_revision_down_trailing_7_days,eps_estimate_revision_up_trailing_30_days,eps_estimate_revision_down_trailing_30_days,revenue_estimate_average,revenue_estimate_high,revenue_estimate_low,revenue_estimate_analyst_count,stock_id
0,0,2026-06-30,fiscal quarter,4.2516,4.7200,3.740,29.0,4.2527,4.2622,4.2604,4.2345,1.0,NaN,1.0,2.0,8.765164e+10,8.954200e+10,8.485700e+10,42.0,2
1,1,2026-03-31,fiscal quarter,4.0706,4.2296,3.943,29.0,4.0643,4.0659,4.0623,3.9621,1.0,NaN,2.0,0.0,8.142614e+10,8.247100e+10,8.100000e+10,43.0,2
2,2,2025-12-31,fiscal quarter,3.8486,4.0300,3.410,33.0,3.8504,3.8504,3.8452,3.8071,1.0,NaN,1.0,0.0,8.027932e+10,8.166103e+10,7.858100e+10,39.0,2
3,3,2025-09-30,fiscal quarter,3.6619,3.7902,3.500,34.0,3.6606,3.6615,3.6612,3.5577,3.0,NaN,4.0,0.0,7.538907e+10,7.664145e+10,7.006600e+10,39.0,2
4,4,2025-06-30,fiscal quarter,3.3794,3.5700,3.310,38.0,3.3759,3.3766,3.3732,3.3128,2.0,NaN,4.0,2.0,7.382683e+10,7.471700e+10,7.257000e+10,42.0,2


In [78]:
df3 = pd.read_csv('./stocks/TSLA/EARNINGS_ESTIMATES.csv')
df3["stock_id"] = 3
df3.head()

,Unnamed: 0,stock_id


In [79]:
df = pd.concat([df1,df2],ignore_index=True)
df.drop("Unnamed: 0",axis=1,inplace=True)
df.head()

,date,horizon,eps_estimate_average,eps_estimate_high,eps_estimate_low,eps_estimate_analyst_count,eps_estimate_average_7_days_ago,eps_estimate_average_30_days_ago,eps_estimate_average_60_days_ago,eps_estimate_average_90_days_ago,eps_estimate_revision_up_trailing_7_days,eps_estimate_revision_down_trailing_7_days,eps_estimate_revision_up_trailing_30_days,eps_estimate_revision_down_trailing_30_days,revenue_estimate_average,revenue_estimate_high,revenue_estimate_low,revenue_estimate_analyst_count,stock_id
0,2027-09-30,fiscal year,9.3883,10.38,8.4300,40.0,9.3279,9.3600,9.3273,9.1317,2.0,NaN,3.0,6.0,5.006671e+11,5.501531e+11,4.738811e+11,41.0,1
1,2026-09-30,fiscal year,8.5227,8.97,7.9600,41.0,8.5065,8.5123,8.5070,8.2624,2.0,NaN,4.0,4.0,4.666955e+11,4.857853e+11,4.487370e+11,41.0,1
2,2026-06-30,fiscal quarter,1.7408,1.89,1.5857,30.0,1.7294,1.7351,1.7325,1.7054,1.0,NaN,3.0,2.0,1.024719e+11,1.093279e+11,9.810500e+10,29.0,1
3,2026-03-31,fiscal quarter,1.9463,2.16,1.5600,32.0,1.9373,1.9554,1.9529,1.8473,1.0,NaN,3.0,2.0,1.096898e+11,1.153689e+11,1.070750e+11,31.0,1
4,2025-12-31,fiscal quarter,2.6708,2.80,2.4500,33.0,2.6731,2.6672,2.6632,2.6509,3.0,NaN,5.0,1.0,1.385221e+11,1.424600e+11,1.366560e+11,32.0,1


In [80]:
df.columns

Index(['date', 'horizon', 'eps_estimate_average', 'eps_estimate_high',
       'eps_estimate_low', 'eps_estimate_analyst_count',
       'eps_estimate_average_7_days_ago', 'eps_estimate_average_30_days_ago',
       'eps_estimate_average_60_days_ago', 'eps_estimate_average_90_days_ago',
       'eps_estimate_revision_up_trailing_7_days',
       'eps_estimate_revision_down_trailing_7_days',
       'eps_estimate_revision_up_trailing_30_days',
       'eps_estimate_revision_down_trailing_30_days',
       'revenue_estimate_average', 'revenue_estimate_high',
       'revenue_estimate_low', 'revenue_estimate_analyst_count', 'stock_id'],
      dtype='str')

In [82]:
earnings_estimates_mapping = {
    'date': 'date',
    'horizon': 'horizon',
    'eps_estimate_average': 'eps_estimate_average',
    'eps_estimate_high': 'eps_estimate_high',
    'eps_estimate_low': 'eps_estimate_low',
    'eps_estimate_analyst_count': 'eps_estimate_analyst_count',
    'eps_estimate_average_7_days_ago': 'eps_estimate_average_7_days_ago',
    'eps_estimate_average_30_days_ago': 'eps_estimate_average_30_days_ago',
    'eps_estimate_average_60_days_ago': 'eps_estimate_average_60_days_ago',
    'eps_estimate_average_90_days_ago': 'eps_estimate_average_90_days_ago',
    'eps_estimate_revision_up_trailing_7_days': 'eps_estimate_revision_up_trailing_7_days',
    'eps_estimate_revision_down_trailing_7_days': 'eps_estimate_revision_down_trailing_7_days',
    'eps_estimate_revision_up_trailing_30_days': 'eps_estimate_revision_up_trailing_30_days',
    'eps_estimate_revision_down_trailing_30_days': 'eps_estimate_revision_down_trailing_30_days',
    'revenue_estimate_average': 'revenue_estimate_average',
    'revenue_estimate_high': 'revenue_estimate_high',
    'revenue_estimate_low': 'revenue_estimate_low',
    'revenue_estimate_analyst_count': 'revenue_estimate_analyst_count',
    'stock_id': 'stock_id'
}

# Pre-export cleanup: Fill NaN with empty string to avoid "invalid input syntax"
df.rename(columns=earnings_estimates_mapping,inplace=True)


In [84]:
df.to_csv('../stock_db/data/earnings_estimates.csv',index=False)